# Support Vector Machines in Python
### In this , we will build a Support Vector Machine for classification using scikit-learn and the Radial Basis Function (RBF) Kernel. Our training dataset contains continuous & categorical data from the UCI Machine Learning Repository to predict whether or not a person will default on their credit card.

Support Vector Machines are one of the best machine learning methods when getting the correct answer is a higher priority than understanding why we get the correctanswer. They work really well with relatively small datasets and they tend to work well "out of the box". In other words , they don't need optimisation.
In this i do : 
1. importing data from a file
2. Missing data: (1. Identifying Missing Data, 2. Dealing with Missing Data)
3. Downsampling Data
4. Formatting the Data for support vector machines:
 (1. Splitting data into Dependent and independent variables
  2. One-Hot-Encoding
  3. Centering and Scaling the Data)
5. Building a Preliminary Support Vector Machine
6. Optimizing Parameters with Cross Validation (Using Cross Validation to find the best values for Gamma and Regularization)
7. Building, Evaluating, Drawing & interpreting the Final Support Vector Machine

In [6]:
import pandas as pd   #for loading and manipulating data and for One-Hot Encoding
import numpy as np    #data manipulation
import matplotlib.pyplot as ply # drawing graphs
import matplotlib.colors as colors
from sklearn.utils import resample #downsample the dataset
from sklearn.model_selection import train_test_split #split data into training and testing sets
from sklearn.preprocessing import  scale #scale and center data
from sklearn.model_selection import GridSearchCV # for crosss validation
from sklearn.metrics import confusion_matrix #creates a confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay  #draws a confusion matrix
from sklearn.decomposition import PCA # to perform PCA to plot the data
!pip install xlrd


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
#df = pd.read_csv("UCI_Credit_Card.csv",
#                 header=1, #to get column names available in second line
#                 sep='\t') # pandas automatically detects delimeters.

# we can also read in the original MS Excel file directly from the website
df = pd.read_excel("https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls",
                   header=1,
                   )

In [10]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [ ]:
df.rename({'default payment next month':'default'}, axis='columns', inplace=True)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [12]:
df.drop('ID', axis=1, inplace=True) ## set axis=0 to remove rows, set axis=1 to remove columns
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


# Missing Data Part1: Identifying Missing Data
Missing Data is simply a blank space, or a surrogate value like NA, that indicates that we failed to collect data for one of the features. For example, if we forgot to ask someone's age, or forgot to write it down, then we would have a blank space in the datasetfor that person's age.

There are two main ways to deal with missing data:

1. We can remove the rows that contain missing data from the dataset. How big of a waste this is depends on how important the missing value is for classification. For example, if we are missing a value fro age, and age is not useful for classifying if people have heart disease or not, then it would be a shame to throw out all of someone's data just because we do not have their age.

2. We can impute the values that are missing. In thsi context impute is just a fancy way of saying "We can make an educated guess about what the value should be." Continuing our example where we are missing a value for age, instead of throwing out the entiire row of data, we can fill the misssing value with the average age or the median age, or use some other, more sophisticated approach, to guess at an appropriate value.